In [5]:
import sys
print(sys.executable)

import torch
print(torch.__version__)


d:\Euphoscan\.venv\Scripts\python.exe
2.7.1+cpu


In [1]:
import torch
print(torch.__version__)


2.7.1+cpu


In [ ]:
import os
import pandas as pd
import zipfile
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from io import BytesIO

class Ham10000Dataset(Dataset):
    def __init__(self, csv_file, zip_file1, zip_file2, transform=None):
        self.df = pd.read_csv(csv_file)
        self.zip1 = zipfile.ZipFile(zip_file1)
        self.zip2 = zipfile.ZipFile(zip_file2)
        self.transform = transform

        self.label_names = sorted(self.df['dx'].dropna().unique())
        self.label_map = {name: idx for idx, name in enumerate(self.label_names)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = row['image_id'] + '.jpg'

        img_data = None
        if filename in self.zip1.namelist():
            img_data = self.zip1.read(filename)
        elif filename in self.zip2.namelist():
            img_data = self.zip2.read(filename)
        else:
            return None  # Skip if missing

        image = Image.open(BytesIO(img_data)).convert('RGB')
        label = self.label_map[row['dx']]

        if self.transform:
            image = self.transform(image)

        return image, label

def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))
    return torch.utils.data.dataloader.default_collate(batch)

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 7),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.classifier(x)
        return x

def main():
    csv_path = r'D:\Euphoscan\data\metadata.csv'
    zip_file1 = r'D:\Euphoscan\data\HAM10000_images_part_1.zip'
    zip_file2 = r'D:\Euphoscan\data\HAM10000_images_part_2.zip'

    model_folder = r'D:\Euphoscan\model'
    model_save_path = os.path.join(model_folder, 'euphoscan_cnn_model.pth')

    print("Checking if files and folders exist:")
    print(f"CSV exists: {os.path.exists(csv_path)}")
    print(f"ZIP 1 exists: {os.path.exists(zip_file1)}")
    print(f"ZIP 2 exists: {os.path.exists(zip_file2)}")
    print(f"Model folder exists: {os.path.exists(model_folder)}")

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV file not found at: {csv_path}")
    if not os.path.exists(zip_file1) and not os.path.exists(zip_file2):
        raise FileNotFoundError(f"Neither zip file exists:\n{zip_file1}\n{zip_file2}")
    if not os.path.exists(model_folder):
        print(f"Creating model folder: {model_folder}")
        os.makedirs(model_folder)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    dataset = Ham10000Dataset(csv_path, zip_file1, zip_file2, transform)

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=32, num_workers=0, collate_fn=collate_fn)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 10
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for batch in train_loader:
            if not batch or len(batch) == 0:
                continue

            images, labels = batch
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_loss = running_loss / total if total > 0 else 0
        train_acc = correct / total if total > 0 else 0
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.4f}")

        # Validation loop
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for batch in val_loader:
                if not batch or len(batch) == 0:
                    continue

                images, labels = batch
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()

        val_loss /= val_total if val_total > 0 else 1
        val_acc = val_correct / val_total if val_total > 0 else 0
        print(f"Epoch {epoch+1}/{epochs} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")

    torch.save(model.state_dict(), model_save_path)
    print(f"Model saved to {model_save_path}")

if __name__ == '__main__':
    main()


Checking if files and folders exist:
CSV exists: True
ZIP 1 exists: True
ZIP 2 exists: True
Model folder exists: True
Using device: cpu
Epoch 1/10 - Train Loss: 1.0087 - Train Acc: 0.6656
Epoch 1/10 - Val Loss: 0.9369 - Val Acc: 0.6725
Epoch 2/10 - Train Loss: 0.8956 - Train Acc: 0.6737
